# 00 — Preparation de donnees unifiee (6 membres)

**Projet** : ReadmitGuard — Prediction de readmission diabetique a 30 jours  
**Dataset** : UCI #296 — Diabetes 130-US Hospitals (1999-2008)  
**Equipe** : Ryma, Fyras, Najd, Emna, Fatma, Faycel  

Ce notebook produit **11 datasets** couvrant les objectifs de tous les membres.  
Aucun objectif n'est privilegie au detriment d'un autre.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)

SEED = 42
np.random.seed(SEED)
print('Imports OK')

## 1. Chargement des donnees brutes

In [ ]:
RAW_PATH = "diabetes+130-us+hospitals+for+years+1999-2008/diabetic_data.csv"

df_raw = pd.read_csv(RAW_PATH, low_memory=False, na_values="?")
print(f"Raw: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} cols")
df_raw.head(2)

## 2. Nettoyage commun (toutes les cibles)

In [ ]:
# 2a. Exclure les patients decedes ou en hospice
# discharge_disposition_id : 11,19,20,21 = expired ; 13,14 = hospice
died_hospice_ids = {11, 13, 14, 19, 20, 21}
mask_alive = ~df_raw["discharge_disposition_id"].isin(died_hospice_ids)
df = df_raw[mask_alive].copy()
print(f"Apres exclusion deces/hospice: {df.shape[0]:,} rows (supprimes: {df_raw.shape[0] - df.shape[0]:,})")

# 2b. Supprimer gender Unknown/Invalid
mask_gender = df["gender"] != "Unknown/Invalid"
print(f"Gender Unknown/Invalid supprimes: {(~mask_gender).sum()}")
df = df[mask_gender].copy()

# 2c. Supprimer la colonne weight (97% manquant)
df.drop(columns=["weight"], inplace=True)

# 2d. Traitement des valeurs manquantes
df["payer_code"].fillna("Missing", inplace=True)
df["medical_specialty"].fillna("Unknown", inplace=True)
df["race"].fillna("Unknown", inplace=True)
df["A1Cresult"].fillna("Not_Tested", inplace=True)
df["max_glu_serum"].fillna("Not_Tested", inplace=True)

# 2e. Supprimer les 13 medicaments quasi-constants (>97% une seule valeur)
med_cols_to_drop = [
    "acetohexamide", "tolbutamide", "troglitazone", "tolazamide",
    "examide", "citoglipton", "glimepiride-pioglitazone",
    "metformin-rosiglitazone", "metformin-pioglitazone",
    "chlorpropamide", "acarbose", "miglitol", "glyburide-metformin"
]
existing = [c for c in med_cols_to_drop if c in df.columns]
df.drop(columns=existing, inplace=True)

print(f"Nettoyage termine. Shape: {df.shape}")
print(f"NaN restants: {df.isna().sum().sum()}")

## 3. Feature Engineering

In [ ]:
# 3a. Groupement ICD-9 (Strack et al., 2014)
def map_icd9_to_group(code):
    if pd.isna(code) or str(code).strip() == "":
        return "Other"
    code = str(code).strip()
    if code.startswith("E") or code.startswith("V"):
        return "External"
    try:
        num = float(code)
    except ValueError:
        return "Other"
    if 390 <= num <= 459 or num == 785:
        return "Circulatory"
    elif 460 <= num <= 519 or num == 786:
        return "Respiratory"
    elif 520 <= num <= 579 or num == 787:
        return "Digestive"
    elif 250 <= num < 251:
        return "Diabetes"
    elif 800 <= num <= 999:
        return "Injury"
    elif 710 <= num <= 739:
        return "Musculoskeletal"
    elif 580 <= num <= 629 or num == 788:
        return "Genitourinary"
    elif 140 <= num <= 239:
        return "Neoplasms"
    else:
        return "Other"

for col in ["diag_1", "diag_2", "diag_3"]:
    df[f"{col}_group"] = df[col].apply(map_icd9_to_group)

print("ICD-9 groupe en 9 familles cliniques")
print(df["diag_1_group"].value_counts())

In [ ]:
# 3b. Age ordinal
age_map = {
    "[0-10)": 0, "[10-20)": 1, "[20-30)": 2, "[30-40)": 3,
    "[40-50)": 4, "[50-60)": 5, "[60-70)": 6, "[70-80)": 7,
    "[80-90)": 8, "[90-100)": 9
}
df["age_ordinal"] = df["age"].map(age_map)

# 3c. Features agregees
df["total_prior_visits"] = (
    df["number_outpatient"] + df["number_emergency"] + df["number_inpatient"]
)

# Nombre de changements de medicaments
med_change_cols = [c for c in df.columns if c not in [
    "encounter_id", "patient_nbr", "readmitted", "age", "age_ordinal",
    "gender", "race", "payer_code", "medical_specialty",
    "admission_type_id", "discharge_disposition_id", "admission_source_id",
    "diag_1", "diag_2", "diag_3", "diag_1_group", "diag_2_group", "diag_3_group",
    "number_outpatient", "number_emergency", "number_inpatient",
    "total_prior_visits", "num_lab_procedures", "num_procedures",
    "num_medications", "number_diagnoses", "time_in_hospital",
    "max_glu_serum", "A1Cresult", "change", "diabetesMed"
] and df[c].dtype == "object"]

def count_med_changes(row):
    return sum(1 for c in med_change_cols if row[c] in ["Up", "Down"])

df["num_med_changes"] = df.apply(count_med_changes, axis=1)

# Binary indicators
df["insulin_prescrite"] = (df["insulin"] != "No").astype(int)
df["metformin_prescrite"] = (df["metformin"] != "No").astype(int)
df["insulin_changed"] = (df["insulin"].isin(["Up", "Down"])).astype(int)

print(f"Features creees. Shape: {df.shape}")

## 4. Construction des cibles

In [ ]:
# 4a. target_readmit (Ryma, Najd A, Faycel 1/6)
df["target_readmit"] = (df["readmitted"] == "<30").astype(int)
print(f"target_readmit: {df['target_readmit'].mean():.1%} positifs")

# 4b. target_change (Emna, Fatma 1)
df["target_change"] = (df["change"] == "Ch").astype(int)
print(f"target_change: {df['target_change'].mean():.1%} positifs")

# 4c. target_insulin (Emna, Fatma 2)
df["target_insulin"] = df["insulin_changed"]
print(f"target_insulin: {df['target_insulin'].mean():.1%} positifs")

# 4d. target_suivi (Fatma 3)
df["suivi_score"] = (
    (df["A1Cresult"] != "Not_Tested").astype(int) +
    (df["max_glu_serum"] != "Not_Tested").astype(int) +
    (df["total_prior_visits"] >= 2).astype(int)
)
df["target_suivi"] = (df["suivi_score"] >= 2).astype(int)
print(f"target_suivi: {df['target_suivi'].mean():.1%} positifs")

# 4e. target_compagnon (Fatma 4)
patient_counts = df.groupby("patient_nbr")["encounter_id"].transform("count")
df["target_compagnon"] = (
    (df["target_readmit"] == 0) &
    (df["num_med_changes"] <= 1) &
    (df["suivi_score"] >= 2)
).astype(int)
print(f"target_compagnon: {df['target_compagnon'].mean():.1%} positifs")

# 4f. target_risque_oubli (Fatma 5)
df["target_risque_oubli"] = (
    (df["num_medications"] >= 7) &
    (df["age_ordinal"] >= 6) &
    (df["number_diagnoses"] >= 5)
).astype(int)
print(f"target_risque_oubli: {df['target_risque_oubli'].mean():.1%} positifs")

# 4g. next_time_in_hospital (Najd B)
df["next_time_in_hospital"] = df["time_in_hospital"]
print(f"next_time_in_hospital: mean={df['next_time_in_hospital'].mean():.1f}")

## 5. Definition des groupes de features par objectif

In [ ]:
ID_COLS = ["encounter_id", "patient_nbr"]
RAW_TARGET_COLS = ["readmitted"]
CONSTRUCTED_TARGETS = [
    "target_readmit", "target_change", "target_insulin",
    "target_suivi", "target_compagnon", "target_risque_oubli",
    "suivi_score", "next_time_in_hospital"
]
RAW_DIAG_COLS = ["diag_1", "diag_2", "diag_3"]
DROP_ALWAYS = ID_COLS + RAW_TARGET_COLS + CONSTRUCTED_TARGETS + ["change"]

all_feature_cols = [c for c in df.columns if c not in DROP_ALWAYS + RAW_DIAG_COLS]
print(f"Total features disponibles: {len(all_feature_cols)}")

PATIENT_FRIENDLY_FEATURES = [
    "age_ordinal", "gender", "race", "time_in_hospital",
    "num_medications", "number_diagnoses", "total_prior_visits",
    "num_lab_procedures", "num_procedures",
    "diag_1_group", "diag_2_group", "diag_3_group",
    "A1Cresult", "max_glu_serum",
    "insulin_prescrite", "metformin_prescrite",
    "diabetesMed", "num_med_changes"
]

TREATMENT_FEATURES = [
    "age_ordinal", "gender", "race", "time_in_hospital",
    "num_medications", "number_diagnoses", "total_prior_visits",
    "num_lab_procedures", "num_procedures",
    "diag_1_group", "diag_2_group", "diag_3_group",
    "A1Cresult", "max_glu_serum",
    "admission_type_id", "discharge_disposition_id",
    "diabetesMed", "num_med_changes",
    "number_outpatient", "number_emergency", "number_inpatient"
]
TREATMENT_FEATURES_CHANGE = TREATMENT_FEATURES + ["insulin_prescrite", "metformin_prescrite"]
TREATMENT_FEATURES_INSULIN = TREATMENT_FEATURES + ["metformin_prescrite"]

SUIVI_FEATURES = [
    "age_ordinal", "gender", "race", "time_in_hospital",
    "num_medications", "number_diagnoses", "total_prior_visits",
    "num_lab_procedures", "num_procedures",
    "diag_1_group", "diag_2_group", "diag_3_group",
    "admission_type_id", "discharge_disposition_id",
    "diabetesMed", "insulin_prescrite", "metformin_prescrite",
    "number_outpatient", "number_emergency", "number_inpatient"
]

COMPAGNON_FEATURES = [
    "age_ordinal", "gender", "race", "time_in_hospital",
    "num_medications", "number_diagnoses", "total_prior_visits",
    "diag_1_group", "diag_2_group",
    "A1Cresult", "max_glu_serum",
    "insulin_prescrite", "metformin_prescrite",
    "diabetesMed", "num_med_changes"
]

RAPPEL_FEATURES = [
    "age_ordinal", "gender", "race",
    "num_medications", "number_diagnoses", "total_prior_visits",
    "diag_1_group", "diag_2_group", "diag_3_group",
    "insulin_prescrite", "metformin_prescrite",
    "diabetesMed", "num_med_changes",
    "number_outpatient", "number_emergency", "number_inpatient"
]

print("Feature groups defined:")
for name, feats in [
    ("all_features", all_feature_cols),
    ("patient_friendly", PATIENT_FRIENDLY_FEATURES),
    ("treatment_change", TREATMENT_FEATURES_CHANGE),
    ("treatment_insulin", TREATMENT_FEATURES_INSULIN),
    ("suivi", SUIVI_FEATURES),
    ("compagnon", COMPAGNON_FEATURES),
    ("rappel", RAPPEL_FEATURES),
]:
    print(f"  {name}: {len(feats)} features")

## 6. Split avec GroupShuffleSplit (anti-fuite patient)

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
groups = df["patient_nbr"].values
train_idx, test_idx = next(gss.split(df, groups=groups))

print(f"Train: {len(train_idx):,} encounters")
print(f"Test:  {len(test_idx):,} encounters")

train_patients = set(df.iloc[train_idx]["patient_nbr"])
test_patients = set(df.iloc[test_idx]["patient_nbr"])
overlap = train_patients & test_patients
assert len(overlap) == 0, f"FUITE: {len(overlap)} patients dans les deux sets!"
print(f"Patients train: {len(train_patients):,}, test: {len(test_patients):,}, overlap: {len(overlap)}")

## 7. Sauvegarde des 11 datasets

In [ ]:
def save_dataset(name, features, target_col, subset_mask=None, description=""):
    """Save train/test split as parquet files."""
    out_dir = Path(f"data/splits/{name}")
    out_dir.mkdir(parents=True, exist_ok=True)
    
    if subset_mask is not None:
        df_sub = df[subset_mask].copy()
        gss_sub = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
        ti, te = next(gss_sub.split(df_sub, groups=df_sub["patient_nbr"].values))
        df_train = df_sub.iloc[ti]
        df_test = df_sub.iloc[te]
    else:
        df_train = df.iloc[train_idx]
        df_test = df.iloc[test_idx]
    
    valid_feats = [f for f in features if f in df.columns]
    
    X_train = df_train[valid_feats]
    X_test = df_test[valid_feats]
    y_train = df_train[[target_col]]
    y_test = df_test[[target_col]]
    groups_tr = df_train[["patient_nbr"]]
    groups_te = df_test[["patient_nbr"]]
    
    X_train.to_parquet(out_dir / "X_train.parquet")
    X_test.to_parquet(out_dir / "X_test.parquet")
    y_train.to_parquet(out_dir / "y_train.parquet")
    y_test.to_parquet(out_dir / "y_test.parquet")
    groups_tr.to_parquet(out_dir / "groups_train.parquet")
    groups_te.to_parquet(out_dir / "groups_test.parquet")
    pd.DataFrame({"feature": valid_feats}).to_csv(out_dir / "features.csv", index=False)
    
    with open(out_dir / "README.txt", "w") as fout:
        fout.write(f"Dataset: {name}\nDescription: {description}\n")
        fout.write(f"Target: {target_col}\nFeatures: {len(valid_feats)}\n")
        fout.write(f"Train: {len(X_train):,}\nTest: {len(X_test):,}\n")
    
    print(f"  {name}: {len(X_train):,} train, {len(X_test):,} test, {len(valid_feats)} feats [{description}]")

print("save_dataset() definie")

In [ ]:
print("=" * 70)
print("SAUVEGARDE DES 11 DATASETS")
print("=" * 70)

# 1. Readmission (Ryma, Najd A, Faycel 1/6)
save_dataset("readmission", all_feature_cols, "target_readmit",
             description="Ryma + Najd A + Faycel 1,6: classification readmission <30j")

# 2. Patient-friendly (Fyras CareStep, Ryma patient tool)
save_dataset("patient_friendly", PATIENT_FRIENDLY_FEATURES, "target_readmit",
             description="Fyras + Ryma: features comprehensibles par le patient")

# 3. Treatment change (Emna, Fatma 1)
save_dataset("treatment_change", TREATMENT_FEATURES_CHANGE, "target_change",
             description="Emna + Fatma 1: prediction changement de traitement")

# 4. Treatment insulin (Emna, Fatma 2) - sans insulin_prescrite = anti-leakage
save_dataset("treatment_insulin", TREATMENT_FEATURES_INSULIN, "target_insulin",
             description="Emna + Fatma 2: prediction changement insuline (anti-leakage)")

# 5. Suivi (Fatma 3)
save_dataset("suivi", SUIVI_FEATURES, "target_suivi",
             description="Fatma 3: prediction qualite de suivi medical")

# 6. Compagnon (Fatma 4) - subset patients multi-encounters
multi_encounter_mask = patient_counts >= 2
save_dataset("compagnon", COMPAGNON_FEATURES, "target_compagnon",
             subset_mask=multi_encounter_mask,
             description="Fatma 4: compagnon digital (patients multi-encounters)")

# 7. Rappel (Fatma 5)
save_dataset("rappel", RAPPEL_FEATURES, "target_risque_oubli",
             description="Fatma 5: risque oubli medicament")

# 8. Najd B - Regression (duree sejour)
readmitted_mask = df["readmitted"] != "NO"
save_dataset("najd_b", all_feature_cols, "next_time_in_hospital",
             subset_mask=readmitted_mask,
             description="Najd B: regression duree sejour (patients readmis)")

# 9. Clustering (Faycel 2+5)
cluster_dir = Path("data/splits/clustering")
cluster_dir.mkdir(parents=True, exist_ok=True)
cluster_features = [
    "age_ordinal", "gender", "race", "time_in_hospital",
    "num_medications", "number_diagnoses", "total_prior_visits",
    "num_lab_procedures", "num_procedures",
    "diag_1_group", "diag_2_group", "diag_3_group",
    "A1Cresult", "max_glu_serum",
    "insulin_prescrite", "metformin_prescrite",
    "diabetesMed", "num_med_changes",
    "admission_type_id", "discharge_disposition_id"
]
df[cluster_features].to_parquet(cluster_dir / "X_full.parquet")
pd.DataFrame({"feature": cluster_features}).to_csv(cluster_dir / "features.csv", index=False)
print(f"  clustering: {len(df):,} rows, {len(cluster_features)} feats [Faycel 2+5: polypharmacie + personas]")

# 10. Equity audit (Faycel 4, Ryma ODS3)
equity_dir = Path("data/splits/equity")
equity_dir.mkdir(parents=True, exist_ok=True)
equity_cols = all_feature_cols + ["target_readmit"]
# Deduplicate while preserving order
seen = set()
equity_cols_unique = []
for c in equity_cols:
    if c not in seen:
        seen.add(c)
        equity_cols_unique.append(c)
df.iloc[train_idx][equity_cols_unique].to_parquet(equity_dir / "train_with_sensitive.parquet")
df.iloc[test_idx][equity_cols_unique].to_parquet(equity_dir / "test_with_sensitive.parquet")
print(f"  equity: train + test saved [Faycel 4 + Ryma ODS3: audit equite]")

# 11. Association rules (Faycel 3)
assoc_dir = Path("data/splits/association")
assoc_dir.mkdir(parents=True, exist_ok=True)
assoc_features = [
    "diag_1_group", "diag_2_group", "diag_3_group",
    "age_ordinal", "gender", "race",
    "insulin_prescrite", "metformin_prescrite",
    "diabetesMed", "num_med_changes",
    "A1Cresult", "max_glu_serum",
    "target_readmit"
]
df[assoc_features].to_parquet(assoc_dir / "data_association.parquet")
print(f"  association: {len(df):,} rows, {len(assoc_features)} feats [Faycel 3: regles association]")

# Common cleaned dataset
common_dir = Path("data/processed")
common_dir.mkdir(parents=True, exist_ok=True)
df.to_parquet(common_dir / "cleaned_common.parquet")
print(f"  cleaned_common: {len(df):,} rows, {df.shape[1]} cols")

print("\n" + "=" * 70)
print("TOUS LES DATASETS SAUVEGARDES AVEC SUCCES")
print("=" * 70)

## 8. Preprocessors (ColumnTransformer) par type de modele

In [ ]:
def get_col_types(feature_list):
    num_cols = [c for c in feature_list if c in df.columns and df[c].dtype in ['int64', 'float64']]
    cat_cols = [c for c in feature_list if c in df.columns and df[c].dtype == 'object']
    return num_cols, cat_cols

num_r, cat_r = get_col_types(all_feature_cols)
preprocessor_readmit = ColumnTransformer([
    ("num", StandardScaler(), num_r),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_r)
], remainder="drop")

num_pf, cat_pf = get_col_types(PATIENT_FRIENDLY_FEATURES)
preprocessor_pf = ColumnTransformer([
    ("num", StandardScaler(), num_pf),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_pf)
], remainder="drop")

num_t, cat_t = get_col_types(TREATMENT_FEATURES_CHANGE)
preprocessor_treat = ColumnTransformer([
    ("num", "passthrough", num_t),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_t)
], remainder="drop")

print("Preprocessors crees:")
print(f"  readmit: {len(num_r)} num + {len(cat_r)} cat")
print(f"  patient_friendly: {len(num_pf)} num + {len(cat_pf)} cat")
print(f"  treatment: {len(num_t)} num + {len(cat_t)} cat")
print()
print("Note: Utilisez class_weight='balanced' plutot que SMOTE pour le desequilibre.")

## 9. Validation finale

In [ ]:
print("=" * 70)
print("VALIDATION FINALE")
print("=" * 70)

expected_dirs = [
    "data/splits/readmission", "data/splits/patient_friendly",
    "data/splits/treatment_change", "data/splits/treatment_insulin",
    "data/splits/suivi", "data/splits/compagnon",
    "data/splits/rappel", "data/splits/najd_b",
    "data/splits/clustering", "data/splits/equity",
    "data/splits/association", "data/processed"
]

all_ok = True
for d in expected_dirs:
    exists = Path(d).exists()
    if not exists: all_ok = False
    files = list(Path(d).glob('*')) if exists else []
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {d} ({len(files)} files)")

print()

# Verify no patient leakage
supervised = [d for d in expected_dirs
              if not any(x in d for x in ["clustering","association","processed","equity"])]
for d in supervised:
    p = Path(d)
    if (p / "groups_train.parquet").exists() and (p / "groups_test.parquet").exists():
        g_tr = set(pd.read_parquet(p / "groups_train.parquet")["patient_nbr"])
        g_te = set(pd.read_parquet(p / "groups_test.parquet")["patient_nbr"])
        leak = g_tr & g_te
        status = "NO LEAK" if len(leak)==0 else f"LEAK: {len(leak)}"
        print(f"  [{status}] {d}")

print()

# Summary table
rows = []
for d in expected_dirs:
    p = Path(d)
    name = p.name
    if (p / "X_train.parquet").exists():
        xt = pd.read_parquet(p / "X_train.parquet")
        xe = pd.read_parquet(p / "X_test.parquet")
        rows.append({"dataset": name, "train": len(xt), "test": len(xe), "features": xt.shape[1]})
    elif (p / "X_full.parquet").exists():
        xf = pd.read_parquet(p / "X_full.parquet")
        rows.append({"dataset": name, "train": len(xf), "test": 0, "features": xf.shape[1]})
    elif (p / "data_association.parquet").exists():
        xa = pd.read_parquet(p / "data_association.parquet")
        rows.append({"dataset": name, "train": len(xa), "test": 0, "features": xa.shape[1]})
    elif (p / "cleaned_common.parquet").exists():
        xc = pd.read_parquet(p / "cleaned_common.parquet")
        rows.append({"dataset": name, "train": len(xc), "test": 0, "features": xc.shape[1]})

print(pd.DataFrame(rows).to_string(index=False))
print(f"\nTotal datasets: {len(rows)}")
print("STATUS:", "ALL OK" if all_ok else "ISSUES FOUND")

## 10. Correspondance membres / datasets

| Membre | Objectif | Dataset | Target |
|--------|----------|---------|--------|
| **Ryma** | Readmission + fairness | `readmission/` + `equity/` | target_readmit |
| **Ryma** | Patient tool | `patient_friendly/` | target_readmit |
| **Fyras** | CareStep | `patient_friendly/` | target_readmit |
| **Najd** | Model A (classification) | `readmission/` | target_readmit |
| **Najd** | Model B (regression) | `najd_b/` | next_time_in_hospital |
| **Emna** | Changement traitement | `treatment_change/` | target_change |
| **Emna** | Changement insuline | `treatment_insulin/` | target_insulin |
| **Fatma 1** | Changement traitement | `treatment_change/` | target_change |
| **Fatma 2** | Changement insuline | `treatment_insulin/` | target_insulin |
| **Fatma 3** | Qualite suivi | `suivi/` | target_suivi |
| **Fatma 4** | Compagnon digital | `compagnon/` | target_compagnon |
| **Fatma 5** | Risque oubli | `rappel/` | target_risque_oubli |
| **Faycel 1** | Classification readmit | `readmission/` | target_readmit |
| **Faycel 2** | Clustering polypharmacie | `clustering/` | (unsupervised) |
| **Faycel 3** | Association rules | `association/` | (unsupervised) |
| **Faycel 4** | Equity audit | `equity/` | target_readmit |
| **Faycel 5** | Personas patients | `clustering/` | (unsupervised) |
| **Faycel 6** | Performance readmit | `readmission/` | target_readmit |

## 11. Exemple d'utilisation

```python
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier

# Charger
X_train = pd.read_parquet('data/splits/readmission/X_train.parquet')
X_test = pd.read_parquet('data/splits/readmission/X_test.parquet')
y_train = pd.read_parquet('data/splits/readmission/y_train.parquet').squeeze()
y_test = pd.read_parquet('data/splits/readmission/y_test.parquet').squeeze()

# Preprocesser
X_train_t = preprocessor_readmit.fit_transform(X_train)
X_test_t = preprocessor_readmit.transform(X_test)

# Entrainer
model = GradientBoostingClassifier(random_state=42)
model.fit(X_train_t, y_train)
print(f'Score: {model.score(X_test_t, y_test):.3f}')
```

In [ ]:
print("Notebook termine. Tous les datasets sont prets dans data/splits/")
print("Chaque membre peut charger son dataset avec pd.read_parquet()")